# 03 — Challengers
Same pipeline, same three-head architecture, same splits as the baseline:

| | Model | Params | Why |
|---|---|---|---|
| **A** | `microsoft/Multilingual-MiniLM-L12-H384` | ~118M (most in the embedding table) | about 3× faster than XLM-R base, same vocabulary |
| **B** | `intfloat/multilingual-e5-small` | ~118M | *Replaces IndicBERT (amended plan).* IndicBERT's 12 languages exclude **Sinhala**, which is 45% of the data (si + singlish). E5 has XLM-R vocabulary coverage of Sinhala/Tamil and strong sentence-level pre-training. |
| **C** | TF-IDF (char + word n-grams) + Logistic Regression | — | Classical reference that shows how much the transformers actually add. CPU only, takes seconds. |

**Runtime:** Colab **GPU (T4)**. Requires notebooks 01 (and ideally 02) to have been run.

## 3.1 Setup

In [1]:
# --- Environment setup (Colab or local) -------------------------------------------------
import os, sys, json, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/TensorForge/training")   # upload the `training/` folder here
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)
else:
    ROOT = Path.cwd().resolve().parent                  # training/notebooks -> training/

sys.path.insert(0, str(ROOT / "src"))
from tfml.paths import Paths
P = Paths(ROOT)
print("ROOT:", P.root, "| raw data:", P.raw_data)

import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from tfml import data as D, engine as E, metrics as M
from tfml.rules import decode_arrays, to_records, check_consistency
print("CUDA:", torch.cuda.is_available())
splits = D.load_processed(P)
data_config = json.loads(P.data_config.read_text())

Mounted at /content/drive
ROOT: /content/drive/MyDrive/TensorForge/training | raw data: /content/drive/MyDrive/TensorForge/training/data
CUDA: True


## 3.2 Reusable pipeline
The pipeline was already factored into `src/tfml/`:
`modeling.build_model(model_key)`, `engine.train_model(cfg, splits, ckpt_root)` and `engine.evaluate(model, loader, df)`.
This wrapper trains a model, checks the business-rule contract and saves metrics in the same format as notebook 02.

In [2]:
def run_transformer(cfg: E.TrainConfig):
    res = E.train_model(cfg, splits, P.checkpoints, resume=True)
    val_df, val_dec = splits["val"], res["val_decoded"]
    assert not check_consistency(to_records(val_dec, val_df.ticket_id.tolist()))
    out = {"model": cfg.model_key, "best_epoch": res["best"]["epoch"], "train_config": cfg.__dict__,
           "dev": res["dev"], "val": res["val"],
           "val_per_language": M.per_language(val_df, val_dec).reset_index().to_dict(orient="records"),
           "checkpoint_mb": E.dir_size_mb(res["out_dir"] / "best" / "model.pt")}
    (P.results / f"{cfg.model_key}_metrics.json").write_text(json.dumps(out, indent=2, default=str))
    pd.DataFrame(res["history"]).to_csv(P.results / f"{cfg.model_key}_history.csv", index=False)
    print(json.dumps({k: round(v, 4) for k, v in res["val"].items()}, indent=1))
    print(M.per_language(val_df, val_dec).round(3).to_string())
    del res["model"]; torch.cuda.empty_cache()
    return res

## 3.3 Challenger A: Multilingual-MiniLM-L12-H384
Same hyperparameters as the baseline except **learning rate 5e-5**. Small 384-d distilled encoders
under-fit at 2e-5 in a 5-epoch budget; 5e-5 is the usual MiniLM fine-tuning rate.

In [3]:
res_minilm = run_transformer(E.TrainConfig(
    model_key="minilm", learning_rate=5e-5, batch_size=16, max_length=data_config["max_length"],
    epochs=5, warmup_ratio=0.1, weight_decay=0.01, loss_weights=(1.0, 0.5, 0.3)))

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

...

[transformers] BertModel LOAD REPORT from: microsoft/Multilingual-MiniLM-L12-H384
Key                 | Status     |  | 
--------------------+------------+--+-
pooler.dense.weight | UNEXPECTED |  | 
pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

/content/drive/MyDrive/TensorForge/training/src/tfml/engine.py:158: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


  epoch 1 step 50/225 loss 3.6129
  epoch 1 step 100/225 loss 3.2633
  epoch 1 step 150/225 loss 3.1074
  epoch 1 step 200/225 loss 3.0171
epoch 1: loss 2.9765 | dev cat acc 0.1800 macroF1 0.0277 | sec F1 0.0000 | urg F1 0.0000 | 25.8s
  -> new best, saved to /content/drive/MyDrive/TensorForge/training/checkpoints/minilm/best
  epoch 2 step 50/225 loss 2.6835
  epoch 2 step 100/225 loss 2.6784
  epoch 2 step 150/225 loss 2.6613
  epoch 2 step 200/225 loss 2.6494
epoch 2: loss 2.6224 | dev cat acc 0.3800 macroF1 0.1467 | sec F1 0.0000 | urg F1 0.0000 | 24.6s
  -> new best, saved to /content/drive/MyDrive/TensorForge/training/checkpoints/minilm/best
  epoch 3 step 50/225 loss 2.2208
  epoch 3 step 100/225 loss 2.1136
  epoch 3 step 150/225 loss 2.0627
  epoch 3 step 200/225 loss 2.0207
epoch 3: loss 2.0009 | dev cat acc 0.6150 macroF1 0.3725 | sec F1 0.0000 | urg F1 0.0000 | 26.5s
  -> new best, saved to /content/drive/MyDrive/TensorForge/training/checkpoints/minilm/best
  epoch 4 step 5

## 3.4 Challenger B: multilingual-e5-small (mean pooling)

In [4]:
res_e5 = run_transformer(E.TrainConfig(
    model_key="me5-small", learning_rate=5e-5, batch_size=16, max_length=data_config["max_length"],
    epochs=5, warmup_ratio=0.1, weight_decay=0.01, loss_weights=(1.0, 0.5, 0.3)))

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                 | Status     |  | 
--------------------+------------+--+-
pooler.dense.weight | UNEXPECTED |  | 
pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  epoch 1 step 50/225 loss 3.5795
  epoch 1 step 100/225 loss 3.2574
  epoch 1 step 150/225 loss 3.0998
  epoch 1 step 200/225 loss 2.9142
epoch 1: loss 2.8142 | dev cat acc 0.6700 macroF1 0.4193 | sec F1 0.0000 | urg F1 0.0000 | 23.0s
  -> new best, saved to /content/drive/MyDrive/TensorForge/training/checkpoints/me5-small/best
  epoch 2 step 50/225 loss 1.8132
  epoch 2 step 100/225 loss 1.6812
  epoch 2 step 150/225 loss 1.5519
  epoch 2 step 200/225 loss 1.4525
epoch 2: loss 1.4064 | dev cat acc 0.9825 macroF1 0.9734 | sec F1 0.0000 | urg F1 0.0833 | 25.6s
  -> new best, saved to /content/drive/MyDrive/TensorForge/training/checkpoints/me5-small/best
  epoch 3 step 50/225 loss 0.9446
  epoch 3 step 100/225 loss 0.8992
  epoch 3 step 150/225 loss 0.8641
  epoch 3 step 200/225 loss 0.8299
epoch 3: loss 0.8154 | dev cat acc 0.9925 macroF1 0.9889 | sec F1 0.5116 | urg F1 0.3390 | 26.4s
  -> new best, saved to /content/drive/MyDrive/TensorForge/training/checkpoints/me5-small/best
  epoch

## 3.5 Challenger C: TF-IDF + Logistic Regression (classical reference)
Character n-grams (`char_wb`, 2–5) cope well with inconsistent Singlish/Tanglish transliteration,
and word n-grams capture key phrases. There's one LR per head. `C` is tuned on dev for category macro-F1.

In [5]:
import time, joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion
from tfml.textproc import prepare_input
from tfml.labels import CATEGORIES, SECONDARY_LABELS

def as_text(df):
    return [" | ".join(prepare_input(c, s, t, "|")) for c, s, t in zip(df.channel, df.subject, df.text)]

def full_log_proba(clf, X, n_classes):
    """predict_proba -> log-prob matrix over the full label space (absent classes ~ -inf)."""
    out = np.full((X.shape[0], n_classes), np.log(1e-9))
    out[:, clf.classes_] = np.log(np.clip(clf.predict_proba(X), 1e-9, 1))
    return out

t0 = time.time()
vec = FeatureUnion([
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True, max_features=300_000)),
    ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True, token_pattern=r"(?u)\b\w+\b")),
])
X = {s: (vec.fit_transform if s == "train" else vec.transform)(as_text(splits[s])) for s in ("train", "dev", "val")}
y = {s: D.encode_labels(splits[s]) for s in X}
print("features:", X["train"].shape, f"{time.time() - t0:.1f}s")

best = None
for C in (1, 4, 16, 64):
    clf = LogisticRegression(C=C, max_iter=3000).fit(X["train"], y["train"]["category"])
    from sklearn.metrics import f1_score
    f = f1_score(y["dev"]["category"], clf.predict(X["dev"]), average="macro")
    print(f"C={C}: dev category macro-F1 {f:.4f}")
    if best is None or f > best[0]:
        best = (f, C, clf)
C = best[1]; clf_cat = best[2]
clf_sec = LogisticRegression(C=C, max_iter=3000).fit(X["train"], y["train"]["secondary"])
clf_urg = LogisticRegression(C=C, max_iter=3000, class_weight="balanced").fit(X["train"], y["train"]["urgency"])

def tfidf_logits(s):
    return (full_log_proba(clf_cat, X[s], len(CATEGORIES)), full_log_proba(clf_sec, X[s], len(SECONDARY_LABELS)),
            full_log_proba(clf_urg, X[s], 2))

tf_res = {}
for s in ("dev", "val"):
    dec = decode_arrays(*tfidf_logits(s))
    tf_res[s] = (M.compute_metrics(splits[s], dec), dec)
print(json.dumps({k: round(v, 4) for k, v in tf_res["val"][0].items()}, indent=1))
print(M.per_language(splits["val"], tf_res["val"][1]).round(3).to_string())

features: (3600, 31869) 5.9s
C=1: dev category macro-F1 0.9733
C=4: dev category macro-F1 0.9828
C=16: dev category macro-F1 0.9842
C=64: dev category macro-F1 0.9842
{
 "n": 800,
 "category_acc": 0.6025,
 "category_macro_f1": 0.6253,
 "category_weighted_f1": 0.5978,
 "secondary_acc": 0.945,
 "secondary_precision": 1.0,
 "secondary_recall": 0.45,
 "secondary_f1": 0.6207,
 "urgency_acc": 0.9762,
 "urgency_precision": 0.9296,
 "urgency_recall": 0.825,
 "urgency_f1": 0.8742,
 "exact_match": 0.5675,
 "ece": 0.1745,
 "mean_confidence": 0.428
}
            n  category_acc  category_macro_f1  secondary_acc  urgency_f1  gap_vs_en
language                                                                            
en        280         0.504              0.546          0.871       0.731      0.000
singlish  200         0.850              0.848          0.985       0.941      0.346
si        160         0.481              0.504          0.994       0.970     -0.022
ta        120         0.600   

In [6]:
# Single-ticket CPU latency for the TF-IDF model (vectorize + 3 LR predictions)
sample = splits["val"].sample(100, random_state=0)
times = []
for i in range(len(sample)):
    t = time.perf_counter()
    Xi = vec.transform(as_text(sample.iloc[i:i + 1])); clf_cat.predict_proba(Xi); clf_sec.predict_proba(Xi); clf_urg.predict_proba(Xi)
    times.append(time.perf_counter() - t)
tfidf_latency = E.percentile_ms(times)

tfidf_dir = P.checkpoints / "tfidf-lr"; tfidf_dir.mkdir(parents=True, exist_ok=True)
joblib.dump({"vectorizer": vec, "category": clf_cat, "secondary": clf_sec, "urgency": clf_urg, "C": C}, tfidf_dir / "model.joblib")
out = {"model": "tfidf-lr", "C": C, "dev": tf_res["dev"][0], "val": tf_res["val"][0],
       "val_per_language": M.per_language(splits["val"], tf_res["val"][1]).reset_index().to_dict(orient="records"),
       "checkpoint_mb": E.dir_size_mb(tfidf_dir / "model.joblib"), "latency_cpu": tfidf_latency}
(P.results / "tfidf-lr_metrics.json").write_text(json.dumps(out, indent=2, default=str))
print(tfidf_latency, f"{out['checkpoint_mb']:.1f} MB")

{'p50_ms': 4.158703499911098, 'p95_ms': 5.143830499980595, 'p99_ms': 5.957113970021056, 'mean_ms': 4.2849282299857805} 5.4 MB


## 3.6 Summary
All checkpoints are in `checkpoints/<model>/best/` and metrics in `results/<model>_metrics.json`.
CPU latency for the transformers is measured in notebook 04 (step 4.4), on the same 2-thread setup
for every model, so the numbers are comparable.

In [7]:
rows = []
for f in sorted(P.results.glob("*_metrics.json")):
    m = json.loads(f.read_text()); v = m["val"]
    rows.append({"model": m["model"], "cat_acc": v["category_acc"], "cat_macro_f1": v["category_macro_f1"],
                 "sec_f1": v["secondary_f1"], "urg_f1": v["urgency_f1"], "exact": v["exact_match"], "ece": v["ece"]})
pd.DataFrame(rows).set_index("model").round(4)

,cat_acc,cat_macro_f1,sec_f1,urg_f1,exact,ece
model,,,,,,
me5-small,0.7062,0.7294,0.4576,0.2292,0.5925,0.0586
minilm,0.5512,0.4225,0.2979,0.0000,0.4675,0.1140
tfidf-lr,0.6025,0.6253,0.6207,0.8742,0.5675,0.1745
xlmr-base,0.8088,0.8136,0.8904,0.9434,0.7938,0.1264
